# Colab — Inferencia clásico sobre test split

Aplica el detector clásico (Selective Search + HOG + SIFT + BoVW + SVM χ²) sobre `test.json` y produce un JSON de predicciones en formato COCO.

Salida: `predictions/classical_test.json` en Drive — listo para alimentar el framework de evaluación común (H7).

**Resume**: cada 100 imgs vuelca el JSON. Si Colab cae, al relanzar carga el JSON existente y salta los `image_id` ya predichos.

## Prerequisitos en Drive

En `/MyDrive/grocery-detection/processed/`:
- `test.json`
- `codebook.pkl`
- `classical_svm.pkl`  (tras hard_neg si lo corriste, o sin él)

Si el directorio `/MyDrive/grocery-detection/predictions/` no existe, se crea solo.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/arturmoret/GroceryTracker.git"
REPO_DIR = "/content/repo"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

sys.path.insert(0, REPO_DIR + "/scripts")
sys.path.insert(0, REPO_DIR + "/src")
os.chdir(REPO_DIR)

from colab_helper import (
    mount_drive, install_deps, setup_dataset,
    sync_from_drive, sync_to_drive, run_script,
)
print("helpers cargados")

In [ ]:
mount_drive()
install_deps()

In [ ]:
setup_dataset()

## Bajar artifacts + predicciones previas (si las hay)

In [ ]:
sync_from_drive([
    "test.json",
    "codebook.pkl",
    "classical_svm.pkl",
])
# Predicciones previas para resume incremental:
sync_from_drive(
    ["classical_test.json"],
    drive_subdir="predictions",
    repo_subdir="reports/predictions",
)

## Inferencia

Para empezar de cero (descartar predicciones previas), descomenta `--rebuild`.

In [ ]:
run_script("scripts/run_classical_infer.py")
# run_script("scripts/run_classical_infer.py", "--rebuild")

## Subir predicciones a Drive

In [ ]:
sync_to_drive(
    ["classical_test.json"],
    drive_subdir="predictions",
    repo_subdir="reports/predictions",
)

## Siguiente paso

Con `classical_test.json` en Drive, ya tienes las predicciones del pipeline clásico listas para alimentar el framework de evaluación (H7) y comparar contra YOLOv8s (H6) cuando exista.